# 🔍 ArtHeist — VIPER Forensic Engine
## Detecting AI-Generated Art with Forensic Precision

> **Heist 2026 Datathon** | Multi-stage computer vision pipeline

This notebook assembles all four VIPER stages:
1. **Track Alpha**: Data Ingestion & Validation
2. **Track Beta**: Forensic EDA + Classical Baseline
3. **Track Gamma**: Deep Learning (EfficientNet-B0) + Evaluation
4. **Interpretability**: Grad-CAM + UMAP

---
Run top to bottom after training: `python src/train.py`

In [ ]:
# === SETUP ===
import sys, os
from pathlib import Path

# Ensure project root is on path
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 6)})
sns.set_style('whitegrid')

from src.config import *
print(f'Project root   : {ROOT}')
print(f'Device         : {DEVICE}')
print(f'AI images dir  : {AI_ART_DIR}')
print(f'Real images dir: {REAL_ART_DIR}')

---
## 🗃️ Stage 1: Track Alpha — Data Ingestion

The foundation of VIPER. We validate both dataset directories,
build PyTorch DataLoaders with train/val/test splits, and inspect
a representative batch.

In [ ]:
from src.dataloader import get_dataloaders, get_full_dataset

train_loader, val_loader, test_loader = get_dataloaders(verbose=True)

# ── Inspect one batch ─────────────────────────────────────────────────────────
images, labels, paths = next(iter(train_loader))
print(f'\nBatch shape  : {images.shape}')
print(f'Label counts : {dict(zip(*np.unique(labels.numpy(), return_counts=True)))}')
print(f'Sample path  : {Path(paths[0]).name}')

In [ ]:
# ── Visualize a sample grid ────────────────────────────────────────────────────
import torchvision

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denormalize(t):
    img = t.permute(1,2,0).numpy() * STD + MEAN
    return np.clip(img, 0, 1)

class_names = {0: 'REAL', 1: 'AI'}
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, ax in enumerate(axes.flatten()):
    if i < len(images):
        ax.imshow(denormalize(images[i]))
        ax.set_title(class_names[labels[i].item()], fontsize=8,
                     color='green' if labels[i]==0 else 'red')
    ax.axis('off')
plt.suptitle('Sample Batch — VIPER Training Data', fontsize=12)
plt.tight_layout()
plt.show()

---
## 🔬 Stage 2: Track Beta — Forensic EDA

**The core innovation.** We apply six forensic feature families to every image,
each targeting a different statistical signature of AI generation:

| Feature Family | What it measures | Why it matters |
|---|---|---|
| Pixel Stats | RGB mean/std | AI images often have unnaturally smooth channel distributions |
| FFT Analysis | Frequency energy ratio | AI images show characteristic spectral artifacts |
| Color Entropy | KMeans palette entropy | AI art tends toward over-saturated, lower-entropy palettes |
| Noise Residuals | Gaussian subtraction | Real cameras introduce sensor noise AI lacks |
| GLCM Texture | Contrast/energy/homogeneity | AI textures are often too uniform |
| Edge Density | Canny edge fraction | AI images may have artificially sharp or soft edges |

In [ ]:
from src.eda import build_feature_matrix, plot_eda_summary

if FEATURE_MATRIX_CSV.exists():
    print(f'Loading cached feature matrix: {FEATURE_MATRIX_CSV}')
    df = pd.read_csv(FEATURE_MATRIX_CSV)
else:
    print('Building feature matrix (this may take a few minutes) ...')
    df = build_feature_matrix()

print(f'\nFeature matrix shape: {df.shape}')
print(f'Class balance:\n{df["label"].value_counts()}')
df.head()

In [ ]:
# ── Distribution plots ─────────────────────────────────────────────────────────
feature_cols = [c for c in df.columns if c not in ('image_path', 'label')]
label_map = {0: 'Real', 1: 'AI-Generated'}
df['class'] = df['label'].map(label_map)

key_features = ['fft_high_freq_energy', 'color_entropy', 'noise_energy',
                'glcm_homogeneity', 'canny_edge_density', 'mean_r']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
colors = {'Real': '#4CAF50', 'AI-Generated': '#E53935'}

for ax, feat in zip(axes.flatten(), key_features):
    for cls, grp in df.groupby('class'):
        ax.hist(grp[feat].dropna(), bins=30, alpha=0.6,
                label=cls, color=colors[cls])
    ax.set_title(feat.replace('_', ' ').title())
    ax.legend(fontsize=8)

plt.suptitle('Forensic Feature Distributions: Real vs AI-Generated', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── FFT ratio analysis ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for cls, grp in df.groupby('class'):
    ax.hist(grp['fft_ratio'].dropna(), bins=40, alpha=0.65,
            label=cls, color=colors[cls])
ax.set_xlabel('FFT High-Frequency / Low-Frequency Energy Ratio')
ax.set_ylabel('Image Count')
ax.set_title('FFT Frequency Analysis: AI vs Real Images')
ax.legend()
plt.tight_layout()
plt.show()

real_fft = df[df['label']==0]['fft_ratio'].mean()
ai_fft   = df[df['label']==1]['fft_ratio'].mean()
print(f'Mean FFT ratio — Real: {real_fft:.4f} | AI: {ai_fft:.4f}')

---
## 📊 Track Beta — Logistic Regression Baseline

Training a Logistic Regression on the six forensic feature families
validates that our hand-crafted signals carry genuine discriminative power.

In [ ]:
import json
from src.baseline import run_baseline

baseline_metrics = run_baseline()

print('\n=== Baseline Performance ===')
for k, v in baseline_metrics.items():
    if k != 'top_3_features':
        print(f'  {k:12s}: {v}')

print('\nTop 3 discriminative features:')
for f in baseline_metrics.get('top_3_features', []):
    print(f"  {f['feature']}: {f['coefficient']:+.4f}")

---
## 🧠 Stage 3: Track Gamma — Deep Learning (EfficientNet-B0)

We fine-tune the last 3 MBConv blocks of EfficientNet-B0 on our binary
classification task. Training produces `checkpoints/best_model.pth`.

> ⚠️ **Run `python src/train.py` before this cell if you haven't already.**

In [ ]:
# ── Load training history ──────────────────────────────────────────────────────
history_path = RESULTS_DIR / 'training_history.json'

if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)

    epochs    = [h['epoch'] for h in history]
    train_acc = [h['acc']   for h in history]
    val_acc   = [h['val_acc'] for h in history]
    val_f1    = [h['val_f1']  for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, train_acc, 'o-', label='Train Acc', color='steelblue')
    ax1.plot(epochs, val_acc,   's-', label='Val Acc',   color='darkorange')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.set_title('Training vs Validation Accuracy'); ax1.legend()

    ax2.plot(epochs, val_f1, 'd-', label='Val F1', color='seagreen')
    best_epoch = epochs[np.argmax(val_f1)]
    ax2.axvline(best_epoch, color='red', linestyle='--', alpha=0.5,
                label=f'Best @ epoch {best_epoch}')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1 Score')
    ax2.set_title('Validation F1 Score'); ax2.legend()

    plt.suptitle('VIPER Training Curves — EfficientNet-B0', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ No training history found. Run: python src/train.py')

---
## 📈 Evaluation — Metrics & Confusion Matrix

In [ ]:
# Load or run evaluation
if EVAL_METRICS_JSON.exists():
    with open(EVAL_METRICS_JSON) as f:
        eval_metrics = json.load(f)
else:
    from src.evaluate import evaluate
    eval_metrics = evaluate()

print('=== EfficientNet-B0 Evaluation Results ===')
metrics_to_show = ['accuracy', 'f1', 'precision', 'recall', 'auc_roc']
for m in metrics_to_show:
    if m in eval_metrics:
        print(f'  {m:12s}: {eval_metrics[m]:.4f}')

# Compare with baseline
if BASELINE_METRICS_JSON.exists():
    with open(BASELINE_METRICS_JSON) as f:
        bl = json.load(f)
    print(f'\n  Baseline F1    : {bl.get("f1", "N/A")}')
    print(f'  EfficientNet F1: {eval_metrics.get("f1", "N/A")}')
    if isinstance(eval_metrics.get('f1'), float) and isinstance(bl.get('f1'), float):
        delta = eval_metrics['f1'] - bl['f1']
        print(f'  Improvement    : {delta:+.4f}')

In [ ]:
# ── Display confusion matrix ───────────────────────────────────────────────────
if CONFUSION_MATRIX_PNG.exists():
    from PIL import Image as PILImage
    img = PILImage.open(CONFUSION_MATRIX_PNG)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.show()
else:
    print('Run python src/evaluate.py to generate the confusion matrix.')

---
## 🎯 Interpretability — Grad-CAM Visual Explanations

Grad-CAM visualizes which image regions caused the model to classify an image
as Real or AI-Generated. Red regions = high activation = decisive evidence.

In [ ]:
# ── Display Grad-CAM gallery ───────────────────────────────────────────────────
from PIL import Image as PILImage

correct_imgs        = sorted(GRADCAM_DIR.glob('correct_*.png'))[:4]
misclassified_imgs  = sorted(GRADCAM_DIR.glob('misclassified_*.png'))[:2]
gallery             = correct_imgs + misclassified_imgs

if gallery:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, img_path in zip(axes.flatten(), gallery):
        img = PILImage.open(img_path)
        ax.imshow(img)
        label = 'Correct ✓' if 'correct' in img_path.stem else 'Misclassified ✗'
        ax.set_title(label, color='green' if '✓' in label else 'red')
        ax.axis('off')
    for ax in axes.flatten()[len(gallery):]:
        ax.axis('off')
    plt.suptitle('Grad-CAM Explanations — VIPER Forensic Engine', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Run python src/visualize.py to generate Grad-CAM gallery.')

---
## 🗺️ UMAP — Feature Space Projection

We extract 1280-dimensional feature vectors from EfficientNet's final pooling layer
and project them into 2D using UMAP. Tight clusters = the model has learned a
genuine forensic boundary.

In [ ]:
# ── UMAP scatter plot ─────────────────────────────────────────────────────────
if UMAP_SCATTER_PNG.exists():
    img = PILImage.open(UMAP_SCATTER_PNG)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('UMAP 2D Projection of EfficientNet-B0 Feature Space')
    plt.show()
elif UMAP_FEATURES_CSV.exists():
    from src.visualize import plot_umap_scatter
    plot_umap_scatter()
    img = PILImage.open(UMAP_SCATTER_PNG)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Run python src/evaluate.py and python src/visualize.py first.')

---
## 🧪 Stretch Goals — Robustness & Domain Transfer

In [ ]:
# ── JPEG Robustness ────────────────────────────────────────────────────────────
if JPEG_ROBUSTNESS_PNG.exists():
    img = PILImage.open(JPEG_ROBUSTNESS_PNG)
    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('JPEG Robustness: Accuracy vs Compression Quality')
    plt.show()

# ── WikiArt confidence ─────────────────────────────────────────────────────────
if WIKIART_CONF_JSON.exists():
    with open(WIKIART_CONF_JSON) as f:
        wikiart = json.load(f)

    if 'per_image' in wikiart:
        confs = [x['p_ai_generated'] for x in wikiart['per_image']]
        plt.figure(figsize=(8, 4))
        plt.hist(confs, bins=30, color='#7B1FA2', alpha=0.75)
        plt.axvline(0.5, color='red', linestyle='--', label='Decision boundary')
        plt.xlabel('P(AI-Generated)')
        plt.ylabel('Image Count')
        plt.title(f'WikiArt Domain Transfer — Confidence Distribution\n'
                  f'Mean={wikiart["mean_conf"]:.3f}  '
                  f'{wikiart["pct_flagged"]:.1%} flagged as AI')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print(f'WikiArt note: {wikiart.get("reason", "")}')

---
## ✅ Results Summary

Complete pipeline run results compiled into a single table.

In [ ]:
rows = []
if BASELINE_METRICS_JSON.exists():
    with open(BASELINE_METRICS_JSON) as f:
        bl = json.load(f)
    rows.append({'Model': 'Logistic Regression (Baseline)',
                 'Accuracy': bl.get('accuracy', '-'),
                 'F1': bl.get('f1', '-'),
                 'AUC-ROC': bl.get('auc_roc', '-')})

if EVAL_METRICS_JSON.exists():
    with open(EVAL_METRICS_JSON) as f:
        em = json.load(f)
    rows.append({'Model': 'EfficientNet-B0 (VIPER)',
                 'Accuracy': em.get('accuracy', '-'),
                 'F1': em.get('f1', '-'),
                 'AUC-ROC': em.get('auc_roc', '-')})

if rows:
    summary_df = pd.DataFrame(rows).set_index('Model')
    display(summary_df.style
            .format('{:.4f}', subset=pd.IndexSlice[:, ['Accuracy','F1','AUC-ROC']])
            .set_caption('VIPER Forensic Engine — Final Results'))
else:
    print('Run the full pipeline to populate this table.')